In [1]:
# Cell 1
import os
import re
import time
import json
import torch
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from dotenv import load_dotenv
from pypdf import PdfReader
from huggingface_hub import login

# Load environment variables
load_dotenv()

# Explicit HF Login to fix the unauthenticated warning
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token)
else:
    print("WARNING: HF_TOKEN not found in .env file.")

# Setup device
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

# Define paths
RAW_DATA_DIR = "../data/raw"
PROCESSED_DATA_DIR = "../data/processed"
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

Using device: mps
PyTorch version: 2.11.0


In [2]:
# Cell 2
def clean_text(text):
    if not text: return ""
    return re.sub(r'\s+', ' ', text.replace('\n', ' ')).strip()

def extract_text_from_nested_pdfs(raw_dir):
    extracted_data = [] # Changed from extracted_pages for clarity
    pdf_files = [os.path.join(r, f) for r, _, fs in os.walk(raw_dir) for f in fs if f.endswith('.pdf')]
    
    for filepath in tqdm(pdf_files, desc="Extracting PDFs"):
        folder_name = os.path.basename(os.path.dirname(filepath))
        filename = os.path.basename(filepath)
        
        # Metadata logic
        if folder_name in ['encyclopedia', 'reference', 'stories']:
            source_type, class_level, subject = folder_name, "all", "general"
        else:
            parts = folder_name.split('_')
            source_type = "textbook"
            class_level = parts[1] if len(parts) >= 2 else "1"
            subject = parts[2] if len(parts) >= 3 else "general"
            
        try:
            reader = PdfReader(filepath)
            for i, page in enumerate(reader.pages):
                text = clean_text(page.extract_text())
                if len(text) > 20:
                    extracted_data.append({
                        "filename": filename,
                        "source_type": source_type,
                        "class_level": class_level,
                        "subject": subject,
                        "page_num": i + 1,
                        "text": text
                    })
        except Exception as e:
            print(f"Error in {filename}: {e}")
            
    return extracted_data

extracted_data = extract_text_from_nested_pdfs(RAW_DATA_DIR)
print(f"Extraction finished. {len(extracted_data)} pages total.")

Extracting PDFs:   0%|          | 0/269 [00:00<?, ?it/s]

EOF marker not found


Error in britannica-student-encyclopedia.pdf: Stream has ended unexpectedly
Extraction finished. 4656 pages total.


In [3]:
# Cell 3
def create_overlapping_chunks(pages_data, chunk_size=200, overlap=20):
    chunks = []
    step_size = chunk_size - overlap
    
    for page in tqdm(pages_data, desc="Chunking"):
        words = page["text"].split()
        for i in range(0, len(words), step_size):
            chunk_words = words[i : i + chunk_size]
            if len(chunk_words) > 10:
                chunks.append({
                    "filename": page["filename"],
                    "source_type": page["source_type"],
                    "class_level": page["class_level"],
                    "subject": page["subject"],
                    "page_num": page["page_num"],
                    "chunk_text": " ".join(chunk_words),
                    "word_count": len(chunk_words)
                })
    return chunks

document_chunks = create_overlapping_chunks(extracted_data)
print(f"Total chunks created: {len(document_chunks)}")

Chunking:   0%|          | 0/4656 [00:00<?, ?it/s]

Total chunks created: 6422


In [4]:
# Cell 4
OUTPUT_FILE = os.path.join(PROCESSED_DATA_DIR, "chunks.json")

print(f"Saving {len(document_chunks)} chunks to {OUTPUT_FILE}...")
start_time = time.time()

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(document_chunks, f, indent=4, ensure_ascii=False)

elapsed_time = time.time() - start_time
print(f"Saved successfully in {elapsed_time:.4f} seconds.")

Saving 6422 chunks to ../data/processed/chunks.json...
Saved successfully in 0.0504 seconds.


In [5]:
# Cell 5
if document_chunks:
    sample_idx = np.random.randint(0, len(document_chunks))
    sample = document_chunks[sample_idx]
    
    print("=== SANITY CHECK: RANDOM CHUNK ===")
    print(f"Class: {sample['class_level']} | Subject: {sample['subject']}")
    print(f"File: {sample['filename']} | Page: {sample['page_num']}")
    print(f"Word Count: {sample['word_count']}")
    print("-" * 50)
    print(sample['chunk_text'])
    print("-" * 50)
else:
    print("No chunks found!")

=== SANITY CHECK: RANDOM CHUNK ===
Class: all | Subject: general
File: english-grammar-and-composition-wren-and-martin.pdf | Page: 5
Word Count: 45
--------------------------------------------------
41. SOME CONJUNCTIONS AND THEIR USES -- 157 42. THE INTERJECTION -- 163 43. THE SAME WORD USED AS DIFFERENT PARTS OF SPEECH -- 163 BOOK II. COMPOSITION PART I ANALYSIS, TRANSFORMATION AND SYNTHESIS 1. ANALYSIS OF SIMPLE SENTENCES -- 169-178 Exercise 1-7 -- 170
--------------------------------------------------
